In [1]:
# Cell 1 — Imports
from pathlib import Path
import sys

PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils.io_utils import load_config, load_model
from gcamp_analysis.experiments.tree import ExperimentTreeBuilder, is_video_dir, print_tree
from gcamp_analysis.video_runner import VideoPipelineRunner
from gcamp_analysis.experiments.processor import ExperimentProcessor
from gcamp_analysis.experiments.io import save_comparisons

In [2]:
config_path = PROJECT_ROOT / "config" / "notebook_config.yaml"
config = load_config(config_path)

print(f"Config: {config_path}")

Config: C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\config\notebook_config.yaml


In [3]:
roi_model, roi_cfg = load_model(config["models"], which="roi")
spike_model, spike_cfg = load_model(config["models"], which="spike")

models = {
    "roi": roi_model,
    "roi_config": roi_cfg,
    "spike": spike_model,
    "spike_config": spike_cfg,
}

runner = VideoPipelineRunner.build(config, models)

print(f"ROI model:   {type(roi_model).__name__}")
print(f"Spike model: {type(spike_model).__name__}")

ROI model:   RandomForestClassifier
Spike model: LogisticRegression


In [4]:
EXPERIMENT_ROOT = Path(r"C:\Users\mzinn1\Desktop\grouping_test\Week 2 10uM")  # TODO: change per experiment
assert EXPERIMENT_ROOT.exists(), f"Experiment root not found: {EXPERIMENT_ROOT}"

builder = ExperimentTreeBuilder(is_video_dir=is_video_dir)
tree = builder.build(EXPERIMENT_ROOT)
print_tree(tree)

└── Week 2 10uM
    ├── 2-1
    ├── 2-1_10uM_2m
    ├── 2-2
    ├── 2-2_10uM_9m
    ├── 2-3
    ├── 2-3_10uM_16m
    └── metrics


In [5]:
processor = ExperimentProcessor(
    runner=runner,
    output_root=EXPERIMENT_ROOT,
)
processor.process_tree(tree, verbose=True)


 Processing: 2-1
  Traces: 1045 ROIs, 3650 frames @ 15.0 Hz
  ROI filter: 985/1045 kept (94.3%)
  Spikes: 7937/95285 kept | neurons 985 → 976
  Grouping (corr+sttc): | corr=93 | sttc=29 | corr_vs_sttc=0.77

 Processing: 2-1_10uM_2m
  Traces: 921 ROIs, 3650 frames @ 15.0 Hz
  ROI filter: 764/921 kept (83.0%)
  Spikes: 4017/77404 kept | neurons 764 → 755
  Grouping (corr+sttc): | corr=74 | sttc=11 | corr_vs_sttc=0.74

 Processing: 2-2
  Traces: 684 ROIs, 3650 frames @ 15.0 Hz
  ROI filter: 593/684 kept (86.7%)
  Spikes: 4296/58071 kept | neurons 593 → 592
  Grouping (corr+sttc): | corr=55 | sttc=12 | corr_vs_sttc=0.80

 Processing: 2-2_10uM_9m
  Traces: 544 ROIs, 3650 frames @ 15.0 Hz
  ROI filter: 455/544 kept (83.6%)
  Spikes: 2120/45828 kept | neurons 455 → 450
  Grouping (corr+sttc): | corr=38 | sttc=9 | corr_vs_sttc=0.78

 Processing: 2-3
  Traces: 420 ROIs, 3650 frames @ 15.0 Hz
  ROI filter: 330/420 kept (78.6%)
  Spikes: 2308/32499 kept | neurons 330 → 328
  Grouping (corr+sttc)

In [6]:
sibling_tables = processor.compare_siblings(tree)

for node_path, df in sibling_tables.items():
    if len(df) >= 2:
        print(f"\nNode: {node_path}")
        print(df.to_string(index=False))


Node: C:\Users\mzinn1\Desktop\grouping_test\Week 2 10uM
       child  n_videos  n_neurons  n_groups  decay_tau_mean_unweighted  decay_tau_var_unweighted  decay_tau_within_unweighted  decay_tau_between_unweighted  half_max_width_seconds_mean_unweighted  half_max_width_seconds_var_unweighted  half_max_width_seconds_within_unweighted  half_max_width_seconds_between_unweighted  rise_slope_mean_unweighted  rise_slope_var_unweighted  rise_slope_within_unweighted  rise_slope_between_unweighted  decay_tau_mean_weighted  decay_tau_var_weighted  decay_tau_within_weighted  decay_tau_between_weighted  half_max_width_seconds_mean_weighted  half_max_width_seconds_var_weighted  half_max_width_seconds_within_weighted  half_max_width_seconds_between_weighted  rise_slope_mean_weighted  rise_slope_var_weighted  rise_slope_within_weighted  rise_slope_between_weighted  spike_frequency_mean_unweighted  spike_frequency_var_unweighted  spike_frequency_within_unweighted  spike_frequency_between_unweighted  sp

In [7]:
save_comparisons(
    root=tree,
    sibling_tables=sibling_tables,
    output_subdir="metrics",
    filename="sibling_comparisons.xlsx",
)